# 06. 모델 비교 + 앙상블

## 목표
- Baseline, SARIMA, Prophet, XGBoost 4개 모델군 비교
- 상품군별 최적 모델 분석
- 가중 평균 앙상블 시도
- Test set 최종 평가 + Gap Analysis

In [ ]:
import sys
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# 한글 폰트 설정
fm._load_fontmanager(try_read_cache=False)
sns.set_style("whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

sys.path.insert(0, "..")

print("Setup complete")

## 1. 전체 모델 결과 로드

In [ ]:
# 모든 결과 CSV 로드
baseline_results = pd.read_csv("../outputs/results/baseline_results.csv")
sarima_results = pd.read_csv("../outputs/results/sarima_results.csv")
prophet_results = pd.read_csv("../outputs/results/prophet_results.csv")
xgboost_results = pd.read_csv("../outputs/results/xgboost_results.csv")

# SARIMA에는 aic 컬럼이 있으므로 공통 컬럼만 사용
common_cols = [
    "model",
    "family",
    "mape",
    "rmse",
    "mae",
    "train_time_sec",
    "predict_time_sec",
]

# Validation 기간 최적 모델만 선택
# Baseline: Naive_7d (가장 일반적)
best_baseline = baseline_results[baseline_results["model"] == "Naive_7d"][common_cols]
# SARIMA: SARIMAX (외생변수 포함)
best_sarima = sarima_results[sarima_results["model"] == "SARIMAX"][common_cols]
# Prophet: Prophet_Tuned
best_prophet = prophet_results[prophet_results["model"] == "Prophet_Tuned"][common_cols]
# XGBoost: XGBoost_Tuned
best_xgboost = xgboost_results[xgboost_results["model"] == "XGBoost_Tuned"][common_cols]

# 통합
all_results = pd.concat(
    [best_baseline, best_sarima, best_prophet, best_xgboost], ignore_index=True
)
print(f"Total results: {len(all_results)} rows (4 models x 5 families)")
print(all_results[["model", "family", "mape"]].to_string(index=False))

## 2. 모델 비교 시각화

In [ ]:
# MAPE 히트맵 (family x model)
pivot = all_results.pivot(index="family", columns="model", values="mape")
# 모델 순서 정렬
model_order = ["Naive_7d", "SARIMAX", "Prophet_Tuned", "XGBoost_Tuned"]
pivot = pivot[[c for c in model_order if c in pivot.columns]]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r", ax=ax, linewidths=0.5)
ax.set_title("MAPE (%) — 상품군 x 모델", fontsize=14)
ax.set_ylabel("")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig("../outputs/figures/21_mape_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(pivot.index))
width = 0.2

for i, model in enumerate(pivot.columns):
    ax.bar(x + i * width, pivot[model], width, label=model, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(pivot.index, rotation=0)
ax.set_ylabel("MAPE (%)")
ax.set_title("모델별 MAPE 비교", fontsize=14)
ax.legend()
ax.axhline(y=12, color="red", linestyle="--", alpha=0.5, label="목표 12%")
ax.legend()
plt.tight_layout()
plt.savefig(
    "../outputs/figures/22_model_comparison_bar.png", dpi=150, bbox_inches="tight"
)
plt.show()

## 3. 상품군별 최적 모델 분석

In [ ]:
# 상품군별 최적 모델 선정
best_per_family = all_results.loc[all_results.groupby("family")["mape"].idxmin()]
best_per_family = best_per_family[
    ["family", "model", "mape", "rmse", "mae"]
].reset_index(drop=True)

print("=== 상품군별 최적 모델 ===")
print(best_per_family.to_string(index=False))
print(f"\n평균 Best MAPE: {best_per_family['mape'].mean():.2f}%")

### 상품군별 최적 모델 해석

각 상품군의 데이터 특성에 따라 최적 모델이 다르다:

- **높은 변동성 상품군 (CLEANING 등)**: 어떤 모델도 MAPE가 높음 → 일별 노이즈가 크고, 신호 대비 잡음 비율이 높음
- **안정적 상품군 (BEVERAGES 등)**: Prophet/XGBoost가 계절성 + 외생변수를 잘 포착
- **트렌드 강한 상품군**: Prophet의 유연한 트렌드 모델링이 유리
- **비선형 패턴**: XGBoost가 lag + rolling 피처 조합으로 복잡한 패턴 포착 가능